# Program 9: Sentiment analysis with a pre-trained Hugging Face model

**Aim.** Use a pre-trained model to analyse sentiment in text for a real-world application: patient feedback at a pharmacy counter.

**Theory.** A sentiment pipeline tokenises the text, runs a fine-tuned transformer (DistilBERT, trained on SST-2 movie reviews) and returns POSITIVE or NEGATIVE with a probability. The model was not trained on pharmacy language, so it must be checked on this domain.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
from transformers import pipeline
clf = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
print(clf.model.config.architectures, clf.model.num_parameters() // 1_000_000, 'M parameters')

['DistilBertForSequenceClassification'] 66 M parameters


In [3]:
feedback = [
  'The pharmacist explained my tablets clearly and I felt reassured.',
  'I waited an hour and nobody told me how to take the medicine.',
  'The new reminder app helped my father remember his doses.',
  'The syrup tasted terrible but it did work.',
  'Very confusing instructions, I am worried I took the wrong dose.',
  'The staff were polite and quick.',
]
for text, r in zip(feedback, clf(feedback)):
    print(f"{r['label']:8s} {r['score']:.3f}  {text}")

POSITIVE 1.000  The pharmacist explained my tablets clearly and I felt reassured.
NEGATIVE 0.999  I waited an hour and nobody told me how to take the medicine.
POSITIVE 0.993  The new reminder app helped my father remember his doses.
POSITIVE 0.993  The syrup tasted terrible but it did work.
NEGATIVE 0.999  Very confusing instructions, I am worried I took the wrong dose.
POSITIVE 0.999  The staff were polite and quick.


In [4]:
neg = [t for t, r in zip(feedback, clf(feedback)) if r['label'] == 'NEGATIVE']
print(len(neg), 'of', len(feedback), 'comments flagged for the pharmacy manager to read:')
for t in neg: print(' -', t)

2 of 6 comments flagged for the pharmacy manager to read:
 - I waited an hour and nobody told me how to take the medicine.
 - Very confusing instructions, I am worried I took the wrong dose.


## Observations

The pipeline gives a label and a confidence in a few lines. Mixed comments ("terrible but it did work") are forced into one label, which is a limit of a two-class model.